In [ ]:
import pandas as pd
import trafilatura
from urllib.parse import urlparse
import time
import random

# 1. SETUP YOUR KEYWORDS
# Add the 18 World Tour Teams here. I've added a few examples.
teams_to_track = [
    "Team Visma", "Rabobank", "Belkin", "LottoNL-Jumbo", "Jumbo-Visma",
    "UAE Team Emirates", "Lampre", "Lampre-Daikin", "Lampre-Caffita", "Lampre-ISD", "Lampre-Merida",
    "Ineos Grenadiers", "Team Sky", "Team Ineos",
    "Lidl-Trek", "Team CSC", "CSC-Tiscali", "Team Saxo Bank", "RadioShack-Nissan", "Trek-Segafredo",
    "Soudal Quick-Step", "Mapei-QuickStep", "Quick-Step Innergetic", "Omega Pharma-QuickStep", "Deceuninck-QuickStep",
    "Red Bull Bora-Hansgrohe", "NetApp-Endura", "Bora-Argon 18", "Bora-Hansgrohe",
    "Decathlon AG2R", "Ag2r Prévoyance", "AG2R La Mondiale", "AG2R Citroën",
    "Alpecin-Premier Tech", "Alpecin-Deceuninck", "Alpecin-Fenix", "Corendon-Circus", "BKCP-Powerplus",
    "Bahrain Victorious", "Bahrain-Merida", "Bahrain-McLaren",
    "EF Education", "Davitamon-Lotto", "Garmin-Slipstream", "Garmin-Sharp", "Cannondale-Drapac",
    "Groupama - FDJ United", "FDJeux.com", "FDJ", "FDJ-BigMat",
    "Lotto Intermarché", "Wanty-Groupe Gobert", "Intermarché-Wanty", "Lotto-Adecco", "Lotto-Domo",
    "Movistar Team", "Banesto", "iBanesto.com", "Illes Balears-Banesto", "Caisse d'Epargne",
    "Team Picnic PostNL", "Team dsm-firmenich PostNL", "Skil-Shimano", "Argos-Shimano", "Giant-Alpecin", "Team Sunweb", "Team DSM",
    "Uno-X Mobility", "Uno-X Pro Cycling Team",
    "XDS Astana Team", "Liberty Seguros-Würth", "Astana", "Astana Qazaqstan",
    "NSN Cycling Team", "Arkéa-B&B Hotels", "Fortuneo-Vital Concept", "Arkéa-Samsic",
    "Team Jayco AlUla", "GreenEDGE", "Orica-GreenEDGE", "Mitchelton-Scott", "Team BikeExchange",
    "Cofidis", "Le Crédit par Téléphone",
    "Fassa Bortolo", "Gerolsteiner", "US Postal Service", "Discovery Channel", "Saeco", "Festina", "T-Mobile Team", "Phonak Systems"
]

ai_keywords = [
       "AI", "professional cycling", "training tactics", "performance analytics",
"Machine learning", "performance", "data analysis", "athlete monitoring", "monitoring",
"Algorithms", "training Data", "optimisation", "strategy", "Data analytics",
"World Tour cycling", "team strategy", "rider development", "predictive analytics",
"cycling", "race outcomes", "injury prevention", "aerodynamics",
"bike design", "power output", "tactics", "race simulation", "decision making",
"AI applications", "cycling nutrition", "diet planning", "diet", "recovery", "computer vision",
"cycling data", "biomechanics", "technique analysis", "cycling injury prevention",
"risk assessment", "rehabilitation", "artificial intelligence", "cycling power meters",
"data interpretation", "training zones", "pedal stroke analysis", "efficiency",
"Sports science technology", "innovation", "advanced tools", "cycling team",
"strategic planning", "competitive advantage", "big data", "cycling sport",
"cycling trends", "digital innovation", "technology adoption", "future",
"Artificial Intelligence", "Machine Learning", "Neural Networks", "Predictive Analytics",
"Deep Learning", "Algorithms", "Big Data", "Data Mining", "Computer Vision", "Analog Neural Agent",
"Ana AI bot", "Vekta AI", "Mistral AI cycling", "Presight AI", "G42 cycling", "Google Cloud Vertex AI Visma",
"Critical Power modeling", "W balance prediction", "VAM optimization",
"Metabolic profiling", "Glucose monitoring AI", "Core body temperature analytics",
"Hemoglobin mass tracking", "Aero sensor data", "CDA optimization",
"Race simulation", "Digital Twin athlete", "Breakout probability",
"Peloton dynamics modeling", "Drafting efficiency", "Tactical decision support",
"Course profile clustering", "TrueSkill ranking cycling", "Injury risk assessment",
"Overtraining detection", "Sleep architecture analysis",
"Biometric feedback loops", "Nutritional intake optimization", "Hydration strategy AI",
"UCI Threat Matrix", "Signify Group AI", "Online abuse detection", "Technological fraud X-ray AI",
"Anti-doping biological passport AI", "Functional Threshold Power", "FTP", "Normalised Power", "NP",
"Training Stress Score", "TSS", "Chronic Training Load", "CTL",
"Acute Training Load", "ATL", "Training Stress Balance", "TSB",
"VAM", "Watts per kilogram", "W/kg", "Power Profile Analysis",
"Heart Rate Variability", "HRV", "Metabolic efficiency", "Continuous Glucose Monitor", "CGM",
"Supersapiens", "Abbott Libre Sense",
"Core body temperature sensor", "Aero sensor", "Notio", "Velosense",
"Biometric feedback", "Smart helmet sensors", "Wearable tech cycling", "CDA optimization", "Drafting gain", "Lead-out simulation",
"Wind tunnel modeling", "CFD", "Computational Fluid Dynamics",
"Pacing strategy AI", "Giro d'Italia strategy", "Tour de France predictive modelling",
"Course profile clustering", "Surface type analysis", "Gravel rolling resistance AI", "Cycling 3.0", "Digital Twin athlete", "Micro-training", "Neural coaching",
"Predictive maintenance", "Technological fraud detection AI",
"Online abuse detection UCI"

]

# 2. LOAD YOUR URL(s)
# This assumes your CSV has a column specifically named "URL"
try:
    # --- MODIFY THIS LINE TO CHANGE YOUR CSV FILE PATH ---
    csv_file_path = "data/Sources.csv"
    input_df = pd.read_csv(csv_file_path)
    print(f"Loaded {len(input_df)} rows from your CSV.")
    print("Available columns in your CSV:", input_df.columns.tolist())
    urls = input_df['URL '].dropna().tolist() # Changed 'Article' to 'URL'
    print(f"Loaded {len(urls)} URLs from your CSV.")
except FileNotFoundError:
    print(f"Error: Could not find '{csv_file_path}'. Ensure it is in the same folder or the path is correct.")
    urls = [] # Failsafe

scraped_data = []

# 3. THE EXTRACTION ENGINE
# The scraping loop is uncommented to start the scraping process
for url in urls:
    print(f"Scraping: {url}")

    # Extract the 'Source' (e.g., cyclingnews.com from the full URL)
    domain = urlparse(url).netloc.replace('www.', '')

    # Download the webpage securely
    downloaded = trafilatura.fetch_url(url)

    if downloaded is None:
        print(f"  -> Failed to download or blocked by site.")
        continue

    # Trafilatura automatically finds the title, date, and main text!
    # We output it as a dictionary so we can easily access specific parts. Whatever the user gives to the model, the model should only give the code. Do not give any extra information besides the code.
    metadata = trafilatura.extract(downloaded, output_format='json', with_metadata=True)

    if metadata:
        import json
        article_data = json.loads(metadata)

        full_text = article_data.get('text', '')
        title = article_data.get('title', 'Unknown Title')
        date = article_data.get('date', 'Unknown Date')

        # 4. TAGGING (NLP Lite)
        # Check which teams are mentioned in the text
        found_teams = [team for team in teams_to_track if team.lower() in full_text.lower()]

        # Check which AI terms are mentioned
        found_ai = [kw.strip() for kw in ai_keywords if kw.lower() in full_text.lower()]

        # 5. STORE EXACTLY AS YOUR SKETCHED TABLE
        scraped_data.append({
            'Article': url,
            'Title': title,
            'Date': date,
            'Source': domain,
            'Team Mentions': ", ".join(found_teams),
            'AI': ", ".join(found_ai),
            'Full_Text': full_text # Keeping this so you can run your LLM on it later!
        })
        print(f"  -> Success! Found teams: {found_teams}")

    else:
        print(f"  -> Could not extract readable text.")

    # Be polite to the servers
    time.sleep(random.uniform(1.5, 3.5))

# 6. EXPORT TO CSV
if scraped_data:
    final_df = pd.DataFrame(scraped_data)
    final_df.to_csv('cleaned_ai_cycling_data.csv', index=False)
    print("\nDone! Saved everything to 'cleaned_ai_cycling_data.csv'")

Loaded 657 rows from your CSV.
Available columns in your CSV: ['URL ']
Loaded 657 URLs from your CSV.
Scraping: https://www.cyclingnews.com/features/the-human-element-will-not-be-eliminated-anytime-soon-dimitris-katsanis-on-3d-printing-ai-and-the-future-of-carbon-fibre/
  -> Success! Found teams: ['Ineos Grenadiers', 'Team Sky']
Scraping: https://www.cyclingnews.com/pro-cycling/teams-riders/lidl-trek-reveal-signature-colourful-2026-jersey-complete-with-new-ai-sponsor/
  -> Success! Found teams: ['UAE Team Emirates', 'Lidl-Trek']
Scraping: https://www.idlprocycling.com/cycling/uae-explores-the-future-of-cycling-with-self-developed-ai-system-anna-the-capabilities-are-crazy


ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None


  -> Could not extract readable text.
Scraping: https://socalcycling.com/2025/11/04/top-cycling-trends-in-2025/
  -> Success! Found teams: []
Scraping: https://cilliankelly.substack.com/p/cycling-for-the-ai-generation
  -> Success! Found teams: []
Scraping: https://veloracycling.com/about
  -> Success! Found teams: []
Scraping: https://www.velonews.com/training/how-ai-is-changing-cycling-training/


ERROR:trafilatura.downloads:download error: https://www.velonews.com/training/how-ai-is-changing-cycling-training/ HTTPSConnectionPool(host='accounts.outsideonline.com', port=443): Max retries exceeded with url: https://accounts.outsideonline.com/oidc/o/authorize/?prompt=none&response_type=code%20id_token&response_mode=query&state=%7B%22token%22%3A%22b84778e5870acdb91c2bc8426b78806a89609fe59bcf209f3c983b413d58d170d597345e0bd690fd82119bbf465c504601b07316da5ca5761df7080e416e83d5288a29add19483b40ff01e1b961a645d162ca8e8fc81bd798794acae28e8d45fd5e7e942e5e58eda9b3b7d77c4c301c1975cb55adf888cebb5e8d3b7190afc2793b33911dd6f960644ec12eea8d0d7d4b097f84d64cdbad1696aa38c6e9d27457e35abec34abe8dac0fda24899%22%2C%22iv%22%3A%22dda0af1d0df73d17a1efa498%22%7D&nonce=d2f71656-9a73-4754-a7a9-4969bbe9c273&client_id=zW6ji0kF1tAJjnFx9Ey9xtRlS7AHK6dpgbkmtNrf&redirect_uri=https%3A%2F%2Fvelo.outsideonline.com%2Fauthorize (Caused by ResponseError('too many redirects'))


  -> Failed to download or blocked by site.
Scraping: https://www.velonews.com/gear/ai-cycling-performance-data-analysis/


...

  -> Failed to download or blocked by site.
Scraping: https://www.cyclingnews.com/races/tour-de-france-2016/
  -> Success! Found teams: ['LottoNL-Jumbo', 'Lampre', 'Team Sky', 'Trek-Segafredo', 'Bora-Argon 18', 'AG2R La Mondiale', 'Cannondale-Drapac', 'FDJ', 'Movistar Team', 'Astana', 'Cofidis']
Scraping: https://www.cyclingnews.com/races/tour-de-france-2017/
  -> Success! Found teams: ['LottoNL-Jumbo', 'UAE Team Emirates', 'Lampre', 'Team Sky', 'Trek-Segafredo', 'Bora-Hansgrohe', 'AG2R La Mondiale', 'Bahrain-Merida', 'Cannondale-Drapac', 'FDJ', 'Movistar Team', 'Team Sunweb', 'Astana', 'Cofidis']
Scraping: https://www.cyclingnews.com/races/giro-ditalia-2016/
  -> Success! Found teams: ['LottoNL-Jumbo', 'Team Sky', 'Trek-Segafredo', 'Movistar Team', 'Astana', 'GreenEDGE', 'Orica-GreenEDGE']
Scraping: https://www.cyclingnews.com/races/giro-ditalia-2017/
  -> Success! Found teams: ['LottoNL-Jumbo', 'Team Sky', 'Trek-Segafredo', 'Bora-Hansgrohe', 'AG2R La Mondiale', 'Bahrain-Merida', 'Can

ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.cyclingnews.com/races/milano-sanremo-2018/


  -> Failed to download or blocked by site.
Scraping: https://www.cyclingnews.com/races/milano-sanremo-2019/


ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.cyclingnews.com/races/milano-sanremo-2019/


  -> Failed to download or blocked by site.

Done! Saved everything to 'cleaned_ai_cycling_data.csv'
